# OrionBelt Semantic Layer — Quickstart (Colab)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralfbecher/orionbelt-semantic-layer/blob/main/examples/quickstart_colab.ipynb)

This notebook demonstrates OrionBelt's semantic layer using the **TPC-H** benchmark dataset
in DuckDB. Everything runs inside Colab — no cloud database needed.

> **Requires Python 3.12+** — In Colab, go to *Runtime > Change runtime type* and select Python 3.12 (or later).

**What you'll see:**

1. Install dependencies and start the API
2. Explore the model: dimensions, measures, metrics, ER diagram
3. Compile OBML queries to SQL across 5 dialects
4. Execute queries and see real results
5. Filters: WHERE, HAVING, and filter groups (OR logic)
6. Multi-fact CFL (Composite Fact Layer) in action
7. Metrics (cross-measure formulas)
8. Dimension exclusion (anti-join)
9. Cumulative metrics (running totals & rolling windows)
10. Period-over-period metrics (YoY growth & MoM change)
11. Filtered measures & ratio metrics
12. Lineage and search

## Step 0: Install Dependencies & Create TPC-H Database

Installs OrionBelt with DuckDB execution support, then generates the TPC-H benchmark tables.

In [ ]:
import sys
v = sys.version_info
if v < (3, 12):
    raise RuntimeError(
        f"Python {v.major}.{v.minor} detected — OrionBelt requires 3.12+.\n"
        "In Colab: Runtime > Change runtime type > Python 3.12"
    )

# Install OrionBelt + DuckDB execution support
# (pip warnings about pre-installed Colab packages like ibis/google-adk are harmless)
import subprocess as _sp
_sp.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--disable-pip-version-check", "--no-warn-conflicts",
    "orionbelt-semantic-layer", "ob-flight-extension", "ob-driver-core",
    "ob-duckdb", "duckdb", "pyarrow", "pygments", "sqlparse",
])

In [ ]:
import gc, os, duckdb

DB_PATH = "tpch.duckdb"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

con = duckdb.connect(DB_PATH)
con.execute("INSTALL tpch"); con.execute("LOAD tpch")
con.execute("CALL dbgen(sf=0.01)")

tables = con.execute(
    "SELECT table_name, estimated_size FROM duckdb_tables() ORDER BY table_name"
).fetchall()
for name, size in tables:
    print(f"  {name:12s}  {size:>8,} rows")
con.close(); del con; gc.collect()
print(f"\nDatabase ready: {DB_PATH}")

## Step 1: Write Model & Start the API

The TPC-H OBML model is written to disk, then the OrionBelt API starts as a background process
in single-model mode. Shortcut endpoints (`/v1/query/sql`, `/v1/query/execute`, etc.) auto-resolve.

In [ ]:
MODEL_YAML = r"""
version: 1.0

dataObjects:
  Regions:
    code: region
    schema: main
    columns:
      Region Key:
        code: r_regionkey
        abstractType: int
      Region Name:
        code: r_name
        abstractType: string

  Nations:
    code: nation
    schema: main
    columns:
      Nation Key:
        code: n_nationkey
        abstractType: int
      Nation Name:
        code: n_name
        abstractType: string
      Nation Region Key:
        code: n_regionkey
        abstractType: int
    joins:
      - joinType: many-to-one
        joinTo: Regions
        columnsFrom: [Nation Region Key]
        columnsTo: [Region Key]

  Customers:
    code: customer
    schema: main
    columns:
      Customer Key:
        code: c_custkey
        abstractType: int
      Customer Name:
        code: c_name
        abstractType: string
      Market Segment:
        code: c_mktsegment
        abstractType: string
      Customer Nation Key:
        code: c_nationkey
        abstractType: int
      Account Balance:
        code: c_acctbal
        abstractType: float
        numClass: non-additive
    joins:
      - joinType: many-to-one
        joinTo: Nations
        columnsFrom: [Customer Nation Key]
        columnsTo: [Nation Key]

  Suppliers:
    code: supplier
    schema: main
    columns:
      Supplier Key:
        code: s_suppkey
        abstractType: int
      Supplier Name:
        code: s_name
        abstractType: string
      Supplier Nation Key:
        code: s_nationkey
        abstractType: int
    joins:
      - joinType: many-to-one
        joinTo: Nations
        secondary: true
        pathName: supplier_nation
        columnsFrom: [Supplier Nation Key]
        columnsTo: [Nation Key]

  Parts:
    code: part
    schema: main
    columns:
      Part Key:
        code: p_partkey
        abstractType: int
      Part Name:
        code: p_name
        abstractType: string
      Manufacturer:
        code: p_mfgr
        abstractType: string
      Brand:
        code: p_brand
        abstractType: string
      Part Type:
        code: p_type
        abstractType: string
      Part Size:
        code: p_size
        abstractType: int
      Container:
        code: p_container
        abstractType: string
      Retail Price:
        code: p_retailprice
        abstractType: float
        numClass: non-additive

  Orders:
    code: orders
    schema: main
    columns:
      Order Key:
        code: o_orderkey
        abstractType: int
      Order Customer Key:
        code: o_custkey
        abstractType: int
      Order Date:
        code: o_orderdate
        abstractType: date
      Order Priority:
        code: o_orderpriority
        abstractType: string
      Order Status:
        code: o_orderstatus
        abstractType: string
      Order Total:
        code: o_totalprice
        abstractType: float
        numClass: additive
    joins:
      - joinType: many-to-one
        joinTo: Customers
        columnsFrom: [Order Customer Key]
        columnsTo: [Customer Key]

  Line Items:
    code: lineitem
    schema: main
    columns:
      Line Order Key:
        code: l_orderkey
        abstractType: int
      Line Part Key:
        code: l_partkey
        abstractType: int
      Line Supplier Key:
        code: l_suppkey
        abstractType: int
      Line Number:
        code: l_linenumber
        abstractType: int
      Quantity:
        code: l_quantity
        abstractType: float
        numClass: additive
      Extended Price:
        code: l_extendedprice
        abstractType: float
        numClass: additive
      Discount:
        code: l_discount
        abstractType: float
        numClass: non-additive
      Tax:
        code: l_tax
        abstractType: float
        numClass: non-additive
      Return Flag:
        code: l_returnflag
        abstractType: string
      Line Status:
        code: l_linestatus
        abstractType: string
      Ship Date:
        code: l_shipdate
        abstractType: date
      Ship Mode:
        code: l_shipmode
        abstractType: string
    joins:
      - joinType: many-to-one
        joinTo: Orders
        columnsFrom: [Line Order Key]
        columnsTo: [Order Key]
      - joinType: many-to-one
        joinTo: Parts
        columnsFrom: [Line Part Key]
        columnsTo: [Part Key]
      - joinType: many-to-one
        joinTo: Suppliers
        columnsFrom: [Line Supplier Key]
        columnsTo: [Supplier Key]
      - joinType: many-to-one
        joinTo: Part Supply
        secondary: true
        pathName: lineitem_partsupp
        columnsFrom: [Line Part Key, Line Supplier Key]
        columnsTo: [PS Part Key, PS Supplier Key]

  Part Supply:
    code: partsupp
    schema: main
    columns:
      PS Part Key:
        code: ps_partkey
        abstractType: int
      PS Supplier Key:
        code: ps_suppkey
        abstractType: int
      Available Qty:
        code: ps_availqty
        abstractType: int
        numClass: additive
      Supply Cost:
        code: ps_supplycost
        abstractType: float
        numClass: non-additive
    joins:
      - joinType: many-to-one
        joinTo: Parts
        columnsFrom: [PS Part Key]
        columnsTo: [Part Key]
      - joinType: many-to-one
        joinTo: Suppliers
        columnsFrom: [PS Supplier Key]
        columnsTo: [Supplier Key]

dimensions:
  Region:
    dataObject: Regions
    column: Region Name
    resultType: string
  Nation:
    dataObject: Nations
    column: Nation Name
    resultType: string
  Customer:
    dataObject: Customers
    column: Customer Name
    resultType: string
  Market Segment:
    dataObject: Customers
    column: Market Segment
    resultType: string
  Order Date:
    dataObject: Orders
    column: Order Date
    resultType: date
  Order Priority:
    dataObject: Orders
    column: Order Priority
    resultType: string
  Order Status:
    dataObject: Orders
    column: Order Status
    resultType: string
  Brand:
    dataObject: Parts
    column: Brand
    resultType: string
  Part Type:
    dataObject: Parts
    column: Part Type
    resultType: string
  Ship Mode:
    dataObject: Line Items
    column: Ship Mode
    resultType: string
  Return Flag:
    dataObject: Line Items
    column: Return Flag
    resultType: string
  Supplier:
    dataObject: Suppliers
    column: Supplier Name
    resultType: string

measures:
  Revenue:
    resultType: float
    aggregation: sum
    expression: "{[Line Items].[Extended Price]} * (1 - {[Line Items].[Discount]})"
    synonyms: [sales, income, turnover]
  Gross Amount:
    columns:
      - dataObject: Line Items
        column: Extended Price
    resultType: float
    aggregation: sum
  Total Discount:
    resultType: float
    aggregation: sum
    expression: "{[Line Items].[Extended Price]} * {[Line Items].[Discount]}"
  Total Quantity:
    columns:
      - dataObject: Line Items
        column: Quantity
    resultType: float
    aggregation: sum
  Order Count:
    columns:
      - dataObject: Orders
        column: Order Key
    resultType: int
    aggregation: count
    distinct: true
  Line Item Count:
    columns:
      - dataObject: Line Items
        column: Line Number
    resultType: int
    aggregation: count
  Total Order Value:
    columns:
      - dataObject: Orders
        column: Order Total
    resultType: float
    aggregation: sum
  Supply Cost:
    resultType: float
    aggregation: sum
    expression: "{[Line Items].[Quantity]} * {[Part Supply].[Supply Cost]}"
  Inventory Value:
    resultType: float
    aggregation: sum
    expression: "{[Part Supply].[Available Qty]} * {[Part Supply].[Supply Cost]}"
  Available Stock:
    columns:
      - dataObject: Part Supply
        column: Available Qty
    resultType: int
    aggregation: sum
  Returned Revenue:
    resultType: float
    aggregation: sum
    expression: "{[Line Items].[Extended Price]} * (1 - {[Line Items].[Discount]})"
    filters:
      - column: {dataObject: Line Items, column: Return Flag}
        operator: equals
        values: [{dataType: string, valueString: "R"}]
    description: Revenue from returned items only

metrics:
  Average Order Value:
    expression: "{[Revenue]} / {[Order Count]}"
  Discount Rate:
    expression: "{[Total Discount]} / {[Gross Amount]}"
  Average Line Price:
    expression: "{[Gross Amount]} / {[Line Item Count]}"
  Gross Margin:
    expression: "({[Revenue]} - {[Supply Cost]}) / {[Revenue]}"
  Return Rate:
    expression: "{[Returned Revenue]} / {[Revenue]}"
  Running Revenue:
    type: cumulative
    measure: Revenue
    timeDimension: Order Date
    description: Running total of revenue over time
  Rolling 3m Revenue:
    type: cumulative
    measure: Revenue
    timeDimension: Order Date
    cumulativeType: sum
    window: 3
    description: 3-month rolling sum of revenue
  Revenue YoY Growth:
    type: period_over_period
    expression: "{[Revenue]}"
    periodOverPeriod:
      timeDimension: Order Date
      grain: month
      offsetGrain: year
      comparison: percentChange
    description: Year-over-year revenue growth rate
  Revenue MoM Change:
    type: period_over_period
    expression: "{[Revenue]}"
    periodOverPeriod:
      timeDimension: Order Date
      grain: month
      offsetGrain: month
      comparison: difference
    description: Month-over-month revenue change in absolute terms
""".strip()

MODEL_PATH = "tpch.obml.yml"
with open(MODEL_PATH, "w") as f:
    f.write(MODEL_YAML)
print(f"Model written: {MODEL_PATH}")

In [ ]:
import atexit, json, os, signal, subprocess, sys, time, urllib.request
from urllib.request import Request, urlopen

API_PORT = 8099
API_BASE = f"http://localhost:{API_PORT}"
_api_process = None

def start_api():
    """Start the OrionBelt API as a background process."""
    global _api_process
    if _api_process is not None and _api_process.poll() is None:
        _api_process.terminate(); _api_process.wait()

    env = {
        **os.environ,
        "QUERY_EXECUTE": "true",
        "DB_VENDOR": "duckdb",
        "DUCKDB_DATABASE": os.path.abspath(DB_PATH),
        "MODEL_FILE": os.path.abspath(MODEL_PATH),
        "API_SERVER_PORT": str(API_PORT),
    }
    log = open("api.log", "w")
    _api_process = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "orionbelt.api.app:app",
         "--host", "0.0.0.0", "--port", str(API_PORT)],
        env=env, stdout=log, stderr=log,
    )
    atexit.register(_api_process.terminate)

    for i in range(120):
        if _api_process.poll() is not None:
            log.flush()
            raise RuntimeError(
                f"API exited with code {_api_process.returncode}\n"
                + open("api.log").read()[-2000:]
            )
        try:
            urllib.request.urlopen(f"{API_BASE}/health", timeout=2)
            print(f"API ready on port {API_PORT} ({i + 1}s)")
            break
        except Exception:
            time.sleep(1)
    else:
        _api_process.terminate()
        log.flush()
        raise RuntimeError("API did not start in time\n" + open("api.log").read()[-2000:])

    schema = api("GET", "/v1/schema")
    dims = len(schema.get("dimensions", []))
    measures = len(schema.get("measures", []))
    metrics = len(schema.get("metrics", []))
    print(f"Model ready: {dims} dimensions, {measures} measures, {metrics} metrics")


def api(method, path, body=None):
    """Minimal HTTP helper — accepts a dict or YAML string as body."""
    import yaml
    url = f"{API_BASE}{path}"
    if isinstance(body, str):
        body = yaml.safe_load(body)
    data = json.dumps(body).encode() if body is not None else None
    headers = {"Content-Type": "application/json"} if data else {}
    req = Request(url, data=data, headers=headers, method=method)
    try:
        with urlopen(req) as resp:
            text = resp.read().decode()
    except urllib.request.HTTPError as e:
        text = e.read().decode()
        print(f"HTTP {e.code}: {text}")
        raise
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return text


start_api()

In [ ]:
import hashlib, yaml, sqlparse
from pygments import highlight as _highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import get_lexer_by_name
from pygments.style import Style
from pygments.token import (Comment, Error, Keyword, Literal, Name, Number,
                            Operator, Punctuation, String, Token)
from IPython.display import display, HTML

# ── VS Code Dark+ color theme ──────────────────────────────────────
class VSCodeDark(Style):
    name = "vscode-dark"
    background_color = "#1e1e1e"
    styles = {
        Token: "#d4d4d4", Comment: "italic #6a9955", Comment.Single: "italic #6a9955",
        Comment.Multiline: "italic #6a9955", Keyword: "bold #569cd6",
        Keyword.DML: "bold #569cd6", Keyword.DDL: "bold #569cd6",
        Keyword.Type: "#4ec9b0", Name: "#d4d4d4", Name.Function: "#d4d4d4",
        Name.Builtin: "#d4d4d4", Name.Tag: "#569cd6", Name.Variable: "#ce9178",
        Name.Attribute: "#9cdcfe", Name.Class: "#4ec9b0", Name.Decorator: "#dcdcaa",
        String: "#ce9178", String.Single: "#ce9178", String.Double: "#ce9178",
        String.Symbol: "#ce9178", Number: "#b5cea8", Number.Integer: "#b5cea8",
        Number.Float: "#b5cea8", Literal: "#ce9178", Literal.String: "#ce9178",
        Literal.Scalar: "#ce9178", Operator: "#d4d4d4", Operator.Word: "bold #569cd6",
        Punctuation: "#d4d4d4", Punctuation.Indicator: "#d4d4d4", Error: "#f44747",
    }

_sql_lexer = get_lexer_by_name("sql")
_yaml_lexer = get_lexer_by_name("yaml")
_json_lexer = get_lexer_by_name("json")
_fmt = HtmlFormatter(style=VSCodeDark, noclasses=True, nowrap=False)

_HEADER_BG, _HEADER_FG = "#252526", "#4ec9b0"
_ROW_EVEN, _ROW_ODD = "#1e1e1e", "#252526"
_CELL_FG, _BORDER, _LABEL_FG = "#d4d4d4", "#3c3c3c", "#569cd6"


def _label(text):
    return (f'<div style="font-family:monospace; font-size:11px; font-weight:600;'
            f' color:{_LABEL_FG}; margin:14px 0 4px 0; text-transform:uppercase;'
            f' letter-spacing:1.5px;">{text}</div>')


def _format_sql(sql):
    formatted = sqlparse.format(sql.strip(), reindent=True, keyword_case="upper",
                                indent_columns=True, indent_width=2)
    lines, out = formatted.splitlines(), []
    for line in lines:
        s = line.lstrip()
        if s.upper().startswith(("LEFT JOIN", "RIGHT JOIN", "INNER JOIN",
                                 "CROSS JOIN", "FULL JOIN", "JOIN ")):
            out.append("  " + line)
        else:
            out.append(line)
    return "\n".join(out)


def show_sql(sql):
    return _label("SQL") + _highlight(_format_sql(sql), _sql_lexer, _fmt)


def show_yaml(yaml_str):
    class _D(yaml.Dumper):
        def increase_indent(self, flow=False, indentless=False):
            return super().increase_indent(flow, False)
    data = yaml.safe_load(yaml_str)
    block = yaml.dump(data, Dumper=_D, default_flow_style=False, sort_keys=False)
    return _label("OBML Query") + _highlight(block.strip(), _yaml_lexer, _fmt)


def show_json(data):
    text = json.dumps(data, indent=2)
    return _label("JSON") + _highlight(text, _json_lexer, _fmt)


def show_table(columns, rows):
    col_names = [c["name"] for c in columns]
    numeric = [isinstance(rows[0][i], (int, float)) if rows else False for i in range(len(col_names))]
    hdr = "".join(
        f'<th style="padding:8px 14px; text-align:{"right" if numeric[i] else "left"};'
        f' border-bottom:2px solid {_LABEL_FG}; color:{_HEADER_FG}; font-size:13px;">{n}</th>'
        for i, n in enumerate(col_names))
    body = []
    for idx, row in enumerate(rows):
        bg = _ROW_EVEN if idx % 2 == 0 else _ROW_ODD
        cells = []
        for i, val in enumerate(row):
            align = "right" if numeric[i] else "left"
            if isinstance(val, float):
                fmt = f"{val:.2%}" if abs(val) < 1 and val != 0 else f"{val:,.2f}"
            elif isinstance(val, int):
                fmt = f"{val:,}"
            else:
                fmt = str(val)
            cells.append(f'<td style="padding:6px 14px; text-align:{align}; color:{_CELL_FG};'
                         f' border-bottom:1px solid {_BORDER}; font-size:13px;">{fmt}</td>')
        body.append(f'<tr style="background:{bg};">{"".join(cells)}</tr>')
    return (_label(f"Result &mdash; {len(rows)} row{'s' if len(rows) != 1 else ''}")
            + f'<table style="border-collapse:collapse; background:{_ROW_EVEN}; border-radius:6px;'
            f' overflow:hidden; margin-bottom:12px; font-family:monospace;">'
            f'<thead><tr style="background:{_HEADER_BG};">{hdr}</tr></thead>'
            f'<tbody>{"".join(body)}</tbody></table>')


def show_mermaid(mermaid_src, height=800):
    uid = "m" + hashlib.md5(mermaid_src.encode()).hexdigest()[:8]
    display(HTML(f"""
<div id="{uid}" style="background:#1e1e1e; border-radius:6px;
     padding:16px; min-height:{height}px; overflow:auto;"></div>
<script type="module">
import mermaid from 'https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs';
mermaid.initialize({{ startOnLoad: false, theme: 'dark' }});
const {{ svg }} = await mermaid.render('{uid}_svg', `{mermaid_src}`);
document.getElementById('{uid}').innerHTML = svg;
</script>"""))


def show_result(result, query=None):
    parts = []
    if query is not None:
        parts.append(show_yaml(query))
    parts.append(show_sql(result["sql"]))
    if result.get("rows"):
        parts.append(show_table(result["columns"], result["rows"]))
    display(HTML("".join(parts)))


print("Display helpers loaded.")

## Step 2: Explore the Model

The TPC-H model defines 12 dimensions, 11 measures (including filtered measures),
and 9 metrics (derived, cumulative, period-over-period, and ratio).
All shortcut endpoints auto-resolve to the loaded model.

In [ ]:
# Dimensions — the business concepts you can group by
dims = api("GET", "/v1/dimensions")
print("Dimensions:")
for d in dims:
    print(f"  {d['name']:20s}  ({d['data_object']}.{d['column']})")

In [ ]:
# Measures — aggregations over fact table columns
print("Measures:")
for m in api("GET", "/v1/measures"):
    agg = m.get("aggregation", "")
    expr = m.get("expression", "")
    detail = expr if expr else agg
    print(f"  {m['name']:20s}  {detail}")

In [ ]:
# Metrics — cross-measure formulas
print("Metrics:")
for m in api("GET", "/v1/metrics"):
    print(f"  {m['name']:20s}  = {m['expression']}")

### ER Diagram

The model's data objects, columns, and join relationships at a glance.

In [ ]:
er = api("GET", "/v1/diagram/er?theme=dark")
show_mermaid(er["mermaid"])

## Step 3: Compile Queries

Write queries using business names — OrionBelt compiles them to optimized,
dialect-specific SQL with automatic joins.

### Star schema: Revenue by Region

In [ ]:
query = """
select:
  dimensions: [Region]
  measures: [Revenue, Order Count]
"""
result = api("POST", "/v1/query/sql?dialect=duckdb", query)
show_result(result, query)

The compile result also includes an **explain plan** — which planner was chosen,
the base object, the join path with reasons, and filter counts.

In [ ]:
display(HTML(show_yaml(yaml.dump(result["explain"], default_flow_style=False, sort_keys=False))))

## Step 4: Execute Queries (Real Data!)

The `/v1/query/execute` endpoint compiles OBML and runs the SQL against the database,
returning actual results.

In [ ]:
query = """
select:
  dimensions: [Region]
  measures: [Revenue, Order Count, Total Quantity]
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

In [ ]:
query = """
select:
  dimensions: [Nation, Market Segment]
  measures: [Revenue]
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

### Filters: WHERE, HAVING, and Filter Groups

In [ ]:
query = """
select:
  dimensions: [Ship Mode]
  measures: [Revenue, Order Count]
where:
  - field: Order Priority
    op: "="
    value: 1-URGENT
having:
  - field: Order Count
    op: ">"
    value: 100
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

### Filter Groups: OR Logic

Use filter groups to combine conditions with OR. Here we find revenue for
orders that are either **urgent** or **high priority**.

In [ ]:
query = """
select:
  dimensions: [Ship Mode]
  measures: [Revenue, Order Count]
where:
  - logic: or
    filters:
      - field: Order Priority
        op: "="
        value: 1-URGENT
      - field: Order Priority
        op: "="
        value: 2-HIGH
order_by:
  - field: Revenue
    direction: desc
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 5: Multi-Fact CFL (Composite Fact Layer)

When a query references measures from **independent fact tables**, OrionBelt
automatically uses the CFL planner. It generates `UNION ALL BY NAME` (DuckDB,
Snowflake) or `UNION ALL` with NULL padding (other dialects) to prevent
fan-trap row multiplication.

Here, **Revenue** comes from `Line Items` and **Available Stock** from `Part Supply` —
two independent facts that share `Parts` as a conformed dimension.

In [ ]:
query = """
select:
  dimensions: [Brand]
  measures: [Revenue, Available Stock, Inventory Value]
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

### Secondary Join Path: `usePathNames`

The CFL above treats **Revenue** (Line Items) and **Available Stock** (Part Supply) as
independent facts because there is no primary join between them.

But TPC-H has a composite foreign key: `lineitem.(l_partkey, l_suppkey)` →
`partsupp.(ps_partkey, ps_suppkey)`. This is modelled as a **secondary join** with
`pathName: lineitem_partsupp`.

Activating it with `usePathNames` makes Part Supply reachable from Line Items,
so both tables resolve through a **single star join** — no CFL needed. This enables
the cross-object measure **Supply Cost** (`SUM(l_quantity * ps_supplycost)`) and
the **Gross Margin** metric.

In [ ]:
query = """
select:
  dimensions: [Brand, Supplier]
  measures: [Revenue, Supply Cost, Gross Margin]
usePathNames:
  - source: Line Items
    target: Part Supply
    pathName: lineitem_partsupp
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 6: Metrics (Cross-Measure Formulas)

Metrics are calculated from measures: `{[Total Discount]} / {[Gross Amount]}`.
OrionBelt resolves the formula and generates the correct SQL expression.

In [ ]:
query = """
select:
  dimensions: [Market Segment]
  measures: [Revenue, Gross Amount, Discount Rate, Average Order Value]
order_by:
  - field: Revenue
    direction: desc
limit: 10
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 7: Dimension Exclusion (Anti-Join)

`dimensionsExclude` inverts a dimension-only query: instead of existing combinations,
it returns combinations that **do not exist**. OrionBelt generates a `CROSS JOIN`
of all distinct dimension values, then subtracts existing pairs via `EXCEPT`.

*Which customers have never bought which brands?*

In [ ]:
# Which customers have never bought which brands?
query = """
select:
  dimensions: [Customer, Brand]
dimensionsExclude: true
order_by:
  - field: Customer
  - field: Brand
limit: 20
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 8: Cumulative Metrics (Running Totals & Rolling Windows)

Cumulative metrics compute **running totals** or **rolling window** aggregations
over a time dimension. OrionBelt wraps the planner output in a CTE with
`SUM(...) OVER (ORDER BY date ROWS ...)` window functions.

- **Running Revenue** — unbounded running total from the first date onward
- **Rolling 3m Revenue** — sum over the current row and the 2 preceding periods

In [ ]:
# Running total of revenue by month
query = """
select:
  dimensions:
    - "Order Date:month"
  measures: [Revenue, Running Revenue]
order_by:
  - field: Order Date
limit: 12
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

In [ ]:
# Rolling 3-month window alongside running total
query = """
select:
  dimensions:
    - "Order Date:month"
  measures: [Revenue, Running Revenue, Rolling 3m Revenue]
order_by:
  - field: Order Date
limit: 12
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 9: Period-over-Period Metrics (YoY Growth & MoM Change)

Period-over-period (PoP) metrics compare each time period against a previous one.
OrionBelt generates a **4-CTE date spine** architecture:

1. `date_range` — auto-discovers MIN/MAX date from the data (with all WHERE filters pushed down)
2. `date_spine` — generates a date series at the configured grain with a previous-period lookup
3. `pop_base` — aggregates measures using the spine as the driver (LEFT JOIN to facts)
4. `pop_compare` — self-joins to compute the comparison (percent change, difference, ratio, or previous value)

No hardcoded date ranges needed — the spine is auto-discovered from the data.

In [ ]:
# Year-over-year revenue growth by month
query = """
select:
  dimensions:
    - "Order Date:month"
  measures: [Revenue, Revenue YoY Growth]
order_by:
  - field: Order Date
limit: 24
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

In [ ]:
# Month-over-month revenue change (absolute difference) by region
query = """
select:
  dimensions:
    - "Order Date:month"
    - Region
  measures: [Revenue, Revenue MoM Change]
order_by:
  - field: Order Date
  - field: Region
limit: 20
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 10: Filtered Measures & Ratio Metrics

Measures can include **filters** that gate the aggregation with `CASE WHEN`.
The TPC-H model defines **Returned Revenue** — same expression as Revenue,
but filtered to `Return Flag = 'R'` (returned items only).

Ratio metrics compose filtered and unfiltered measures:
`Return Rate = Returned Revenue / Revenue`.

OrionBelt compiles this to:
```sql
SUM(CASE WHEN "Return Flag" = 'R' THEN "Extended Price" * (1 - "Discount") END)
  / SUM("Extended Price" * (1 - "Discount"))
```

No special syntax needed — just add `filters:` to any measure definition.

In [ ]:
# Returned Revenue (filtered) vs total Revenue, plus Return Rate ratio metric
query = """
select:
  dimensions: [Region]
  measures: [Revenue, Returned Revenue, Return Rate]
order_by:
  - field: Revenue
    direction: desc
"""
result = api("POST", "/v1/query/execute?dialect=duckdb", query)
show_result(result, query)

## Step 11: Lineage & Search

In [ ]:
explain = api("GET", "/v1/explain/Revenue")
display(HTML(show_json(explain)))

In [ ]:
# Search for anything related to "order"
results = api("POST", "/v1/find", {"query": "order"})
for item in results["results"]:
    print(f"  [{item['type']:10s}]  {item['name']}")

---

## Next Steps

- **Gradio UI** — Visual model explorer with ER diagrams, SQL preview, and query builder:
  ```bash
  docker pull ralforion/orionbelt-ui
  docker run -p 7860:7860 -e API_BASE_URL=http://host.docker.internal:8080 ralforion/orionbelt-ui
  ```
- **Arrow Flight SQL** — Connect DBeaver, Tableau, or Power BI directly — see [docs/drivers.md](https://github.com/ralfbecher/orionbelt-semantic-layer/blob/main/docs/drivers.md)
- **DB-API 2.0 Drivers** — Use `ob-duckdb`, `ob-postgres`, `ob-snowflake`, etc. from Python
- **MCP Server** — Connect AI assistants via [orionbelt-semantic-layer-mcp](https://github.com/ralfbecher/orionbelt-semantic-layer-mcp)
- **Full documentation** — [ralforion.com/orionbelt-semantic-layer](https://ralforion.com/orionbelt-semantic-layer/)